# Calo Pipeline Tutorial

This notebook explains the current ColliderFM pipeline step by step.

The goal is to show, in the simplest possible way, how a raw ColliderML calorimeter event becomes model inputs, model outputs, a training loss, and saved artifacts.

Sections:
1. setup
2. load one raw ColliderML event
3. build a calo-only point view
4. create global views
5. create local and masked student views
6. batch several events
7. run the small student/teacher model
8. compute the loss
9. save checkpoints and export embeddings


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from collider_fm.data import ColliderMLDataset
from collider_fm.views import (
    CALO_TYPE_NAMES,
    augment_point_view,
    build_distillation_views,
    build_point_view_from_event,
    local_crop_point_view,
    mask_point_view,
)
from collider_fm.model import create_small_panda_model
from collider_fm.diagnostics import encode_view

plt.style.use('default')


## 1. Choose a split and device

The current PTv3 plus spconv stack needs CUDA for actual model execution, but the dataset and point-view steps can still be explored on CPU.

In [ ]:
split = 'train[:1]'
dataset_type = 'ttbar'
pu_config = 'pu0'
cache_dir = '/mnt/ceph/users/ewulff/data/hf'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 2. Load one raw ColliderML event

The dataset loader returns one event as a dictionary. In this phase we only request `calo_hits`.

In [ ]:
dataset = ColliderMLDataset(
    split=split,
    dataset_type=dataset_type,
    pu_config=pu_config,
    object_types=['calo_hits'],
    cache_dir=cache_dir,
)
event = dataset[0]
calo_hits = event['calo_hits']

print('fields:', sorted(calo_hits.keys()))
print('number of hits:', len(calo_hits['x']))
print('first detector ids:', torch.as_tensor(calo_hits['detector'])[:10].tolist())
print('first energies:', torch.as_tensor(calo_hits['energy'])[:10].tolist())


## 3. Plot the raw calorimeter hits

This is just a quick visual check of the raw event before any preprocessing.

In [ ]:
x = torch.as_tensor(calo_hits['x']).numpy()
z = torch.as_tensor(calo_hits['z']).numpy()
energy = torch.as_tensor(calo_hits['energy']).numpy()

plt.figure(figsize=(7, 5))
plt.scatter(z, x, c=energy, s=3, cmap='inferno', alpha=0.6)
plt.xlabel('z [mm]')
plt.ylabel('x [mm]')
plt.title('Raw calorimeter hits')
plt.colorbar(label='energy')
plt.show()


## 4. Build one point view

This is the central representation in the current code.

Each point keeps:
- `coord`: xyz coordinates
- `feat[:, 0]`: deposited energy
- `feat[:, 1]`: `0` for ECal and `1` for HCal
- `hidden_mask`: points whose input energy was hidden from the student
- `loss_mask`: points that should contribute to the student loss
- `offset`: event boundaries for batched point clouds


In [ ]:
view = build_point_view_from_event(event, device=torch.device('cpu'), max_calo_hits=256)
for key, value in view.items():
    print(key, tuple(value.shape), value.dtype)


## 5. Inspect a few points

This makes the feature contract concrete.

In [ ]:
for i in range(5):
    print(
        i,
        'coord =', view['coord'][i].tolist(),
        'energy =', float(view['energy'][i]),
        'detector_id =', int(view['detector_id'][i]),
        'calo_type =', CALO_TYPE_NAMES[int(view['calo_type'][i].item())],
    )


## 6. Make global views

The current global augmentations are intentionally simple:
- rotation around the beam axis
- small coordinate noise
- small energy jitter

The point order stays the same, which keeps the student/teacher logic easy to understand.

In [ ]:
global_a = augment_point_view(view)
global_b = augment_point_view(view)
print('base:', view['coord'].shape, view['feat'].shape)
print('global_a:', global_a['coord'].shape, global_a['feat'].shape)
print('global_b:', global_b['coord'].shape, global_b['feat'].shape)

plt.figure(figsize=(7, 5))
plt.scatter(view['coord'][:, 2], view['coord'][:, 0], s=3, alpha=0.4, label='base')
plt.scatter(global_a['coord'][:, 2], global_a['coord'][:, 0], s=3, alpha=0.4, label='global_a')
plt.xlabel('z')
plt.ylabel('x')
plt.legend()
plt.title('Base and global views')
plt.show()


## 7. Make local and masked student views

Both student-only views keep the same point order.

- A local view keeps one neighborhood for the loss and hides the rest.
- A masked view hides random points and computes loss only on those hidden points.

In [ ]:
local_view = local_crop_point_view(global_a, keep_fraction=0.5)
masked_view = mask_point_view(global_a, mask_fraction=0.3)

print('local kept for loss:', int(local_view['loss_mask'].sum().item()))
print('local hidden from student:', int(local_view['hidden_mask'].sum().item()))
print('masked loss points:', int(masked_view['loss_mask'].sum().item()))
print('masked hidden points:', int(masked_view['hidden_mask'].sum().item()))


In [ ]:
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.bar(['used', 'ignored'], [local_view['loss_mask'].sum().item(), (~local_view['loss_mask']).sum().item()])
plt.title('Local view loss mask')
plt.subplot(1, 2, 2)
plt.bar(['hidden', 'visible'], [masked_view['hidden_mask'].sum().item(), (~masked_view['hidden_mask']).sum().item()])
plt.title('Masked view hidden points')
plt.tight_layout()
plt.show()


## 8. Batch multiple events

The training code uses several events at once. Each batched view stores event boundaries in `offset`.

In [ ]:
two_event_dataset = ColliderMLDataset(
    split='train[:2]',
    dataset_type=dataset_type,
    pu_config=pu_config,
    object_types=['calo_hits'],
    cache_dir=cache_dir,
)
batch_events = [two_event_dataset[0], two_event_dataset[1]]
views = build_distillation_views(
    batch_events,
    device=torch.device('cpu'),
    max_calo_hits=256,
    add_local_view=True,
    add_masked_view=True,
)
print('number of batch views:', len(views))
for index, batch_view in enumerate(views):
    print(index, batch_view['coord'].shape, batch_view['feat'].shape, batch_view['offset'].tolist())


## 9. Run the model

The current PTv3 plus spconv stack needs CUDA. This cell skips itself on CPU.

In [ ]:
if device.type != 'cuda':
    print('CUDA is not available in this session, so the model forward pass is skipped.')
else:
    model = create_small_panda_model(device=device)
    gpu_views = build_distillation_views(
        batch_events,
        device=device,
        max_calo_hits=256,
        add_local_view=True,
        add_masked_view=True,
    )
    student_outputs, teacher_outputs = model(gpu_views)
    print('student view 0 logits:', tuple(student_outputs[0].shape))
    print('teacher view 0 logits:', tuple(teacher_outputs[0].shape))


## 10. Compute the loss

The current loss is a simple point-level student/teacher cross-entropy over prototype distributions.

Each student view carries a `loss_mask` that says which points should count for that view.

In [ ]:
if device.type != 'cuda':
    print('Loss example skipped because CUDA is not available.')
else:
    loss_masks = [view['loss_mask'] for view in gpu_views]
    loss = model.distillation_loss(student_outputs, teacher_outputs, loss_masks=loss_masks)
    print('loss =', float(loss))
    model.update_center(teacher_outputs)
    model.update_teacher(momentum=0.99)
    print('updated teacher and center')


## 11. Inspect one encoded view

`encode_view` is a helpful diagnostics helper. It returns point features, pooled event embeddings, and prototype logits.

In [ ]:
if device.type != 'cuda':
    print('Encoding example skipped because CUDA is not available.')
else:
    gpu_view = build_point_view_from_event(event, device=device, max_calo_hits=256)
    encoding = encode_view(model, gpu_view)
    for key, value in encoding.items():
        print(key, tuple(value.shape))


## 12. Save checkpoints and export embeddings

The repository scripts do this for you.

```bash
uv run python scripts/train.py --num-epochs 1 --max-train-batches 1 --max-val-batches 1 --add-local-view --add-masked-view
uv run python scripts/export_embeddings.py --checkpoint runs/<run-name>/checkpoint.pt
```

`train.py` saves `checkpoint.pt` in the run directory. `export_embeddings.py` writes a `.pt` file with coordinates, energies, detector IDs, per-point backbone features, and pooled embeddings.

## 13. How everything fits together

- `src/collider_fm/data.py`: loads ColliderML calo tables
- `src/collider_fm/views.py`: turns one event into a point view and makes global, local, and masked student views
- `src/collider_fm/model.py`: defines the student/teacher model and loss
- `scripts/train.py`: runs a small training loop and saves checkpoints
- `scripts/plot_diagnostics.py`: saves quick plots and summary stats
- `scripts/export_embeddings.py`: exports frozen embeddings

If you want to understand the codebase, `data.py`, `views.py`, and `model.py` are the best three files to read first.